
# Introduction

This documentation outlines the method for calculating spatially
standardized latitudinal extinction gradients across stage boundaries
using fossil occurrence data from the Paleobiology Database (PBDB).
Consideration is made for cleaning and filtering data and documenting
each step per [Ten simple rules to follow when cleaning occurrence data
in
palaeobiology](https://tenrules.palaeoverse.org/)[@jonesTenSimpleRules2025].


# Setup

In [3]:
rpkg <- c(
  # Data manipulation
  "dplyr", "readr", "tidyr", "purrr", "conflicted",# "tibble",  "forcats", 
  
  # Geospatial data manipulation

  # "terra", "rgplates", "icosa", "chronosphere", "via", "ncdf4",
  
  # # Accessing and cleaning paleontological occurrence data
  "CoordinateCleaner", "fossilbrush", "divvy",# "divDyn", "paleobioDB", "CoordinateCleaner", "fossilbrush", "divvy",
  
  # # Plotting and visualization
   "ggplot2", "RColorBrewer", "DT")# "ggpubr", "patchwork", "ggstance", "RColorBrewer", "DT","viridis","cowplot","grid",
  
  # # Sharing and publishing
  # "piggyback", "downloadthis")

lapply(rpkg, function(pkg) {
  tryCatch(
    library(pkg, character.only = TRUE)
  )
})|>invisible()


conflicted::conflict_prefer(name = "filter", winner = "dplyr",losers = "stats")

# color blind friendly palette: okabe ito
okabe_ito_10 <- c(
  "#E69F00", # orange
  "#56B4E9", # sky blue
  "#009E73", # bluish green
  "#F0E442", # yellow
  "#0072B2", # blue
  "#D55E00", # vermillion
  "#CC79A7", # reddish purple
  "#999999", # grey
  "#E1BE6A", # tan
  "#40B0A6"  # teal
)

The legacy packages maptools, rgdal, and rgeos, underpinning the sp package,
which was just loaded, will retire in October 2023.
Please refer to R-spatial evolution reports for details, especially
https://r-spatial.org/r/2023/05/15/evolution4.html.
It may be desirable to make the sf package available;
package maintainers should consider adding sf to Suggests:.
The sp package is now running under evolution status 2
     (status 2 uses the sf package in place of rgdal)

[conflicted] Removing existing preference.
[conflicted] Will prefer dplyr::filter over stats::filter.



## Custom Functions

Latitudinally bin occurrence data

In [4]:
#df is cleaned occurence dataframe, lower is list of lower latitude boundaries, 
#upper is the upper boundaries and mid are mid points for plotting purposes. T
#his function makes a copy of original df and adds the latbin column so if you are only wanting 
#to plot latitudinal distributions of taxa you can assign the midpoint lat it will plot data against (See Other Data Exploration section)

#use with divvy::bandit latitudinal subsampling
filter_bin <- function(df, lower, upper, mid) { 
  df %>%
    filter(paleolat >= lower, paleolat < upper) %>%
    mutate(latbin = mid) %>%
    distinct(accepted_name, .keep_all = TRUE)}


# use fitler_bin_hexa recommended only for some subsampling rountine like divvy:cookies
filter_bin_hexa <- function(df, lower, upper, mid) { 
  df %>%
    filter(hexa_lat >= lower, hexa_lat < upper) %>%
    mutate(latbin = mid) %>%
    distinct(accepted_name, .keep_all = TRUE)}

Function to calculate the percentage of genera per latitude bin that went extinct at the end of a given stage.

In [5]:
calc_extinction <- function(fossils, age) {
  
  total_genera <- fossils %>%
    filter(early_interval == age) %>%
    group_by(latbin) %>%
    summarize(num_genera = n_distinct(accepted_name), .groups = "drop")
  
  # Count extinct genera by exact paleolatitude
  num_extinct <- fossils %>%
    filter(early_interval == age, ex == 1) %>% # this line selects genera that went extinct at stage boundy. To be explained in cleaning section.
    group_by(latbin) %>%
    summarize(num_extinct_genera = n_distinct(accepted_name), .groups = "drop")
  
  # Merge totals and extinctions
  percent_extinct <- merge(num_extinct, total_genera, by = "latbin")
  
  # Calculate extinction percentage
  percent_extinct$percent <- round(
    percent_extinct$num_extinct_genera / percent_extinct$num_genera * 100
  )
  
  return(percent_extinct)
}

Build a hexagonal grid and re-project occurrence data to this grid

In [6]:
cell_summary <- function(data, hexa, min_occ = NULL) {
  
  # Get coords and ensure numeric
  coords <- data |> 
    select(paleolng, paleolat) |>
    mutate(across(c(paleolng, paleolat), as.numeric))
  
  # Locate cells
  coords <- coords |> mutate(cell = locate(hexa, coords))
  
  # Count occurrences per cell
  cell_counts <- coords |> 
    group_by(cell) |> 
    summarise(occurrences = n(), .groups = "drop")
  
  # Optionally filter by min_occ
  if (!is.null(min_occ)) {
    
   dropped <- cell_counts %>% filter(occurrences < min_occ)
  
  # Print removed cells
  if (nrow(dropped) > 0) {
    cat("The cells that were flagged are:\n")
    print(dropped$cell)
  } else {
    cat("No cells were flagged.\n")
  }
  
  # Keep only cells meeting min_occ
  cell_counts <- cell_counts %>% filter(occurrences >= min_occ)
  }
  
  return(cell_counts)
}